In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import Galactic, ICRS
from astropy import units as u
from modules.vr_opt import VrOpt
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import MinMaxScaler

import time
from sklearn.cluster import KMeans

import seaborn as sns

import plotly.graph_objs as go
import matplotlib.colors as mcolors

from scipy.spatial import KDTree
import distinctipy
import random

## Data for testing 1

In [ ]:
alphas = [0.9, 0.9, 0.3]
colors = ['tab:red', 'tab:blue', 'tab:grey']
zorders = [1, 1, 0]
log = True

# Create 6D Gaussian data
n = 1000 #1000
X_bg = (np.random.rand(n*5, 6) - 0.5) * 50

mu = np.array([-5, 5, 5, 5, -5, 5])
c_xx = c_yy = 20
c_xy = 15
c_zz = 3
c_uvw = 1
cov = np.diag([c_xx, c_yy, c_zz, c_uvw, c_uvw, c_uvw])
# Put in nonzero covariances in X-Y space
cov[0, 1] = cov[1, 0] = c_xy
# Add signal
X_sig_1 = np.random.multivariate_normal(mu, cov, n//2)

# Second cluster
mu_2 = np.array([5, -5, -5, -5, 5, -5])
c_xx_2 = c_yy = 20
c_xy_2 = -15
c_zz_2 = 3
cov = np.diag([c_xx_2, c_yy, c_zz_2, c_uvw, c_uvw, c_uvw])
# Put in nonzero covariances in X-Y space
cov[0, 1] = cov[1, 0] = c_xy_2
X_sig_2 = np.random.multivariate_normal(mu_2, cov, n//2)

X = np.concatenate([X_bg, X_sig_1, X_sig_2], axis=0)
labels = np.concatenate([np.zeros(n*5), np.ones(n//2), np.ones(n//2)*2])

df = pd.DataFrame(X, columns=['x', 'y', 'z', 'vx', 'vy', 'vz'])
# Transform to spherical coordinates
gal_coords = Galactic(
    u=df.x.values * u.pc,
    v=df.y.values * u.pc,
    w=df.z.values * u.pc,
    # velocities UVW
    U=df.vx.values * u.km / u.s,
    V=df.vy.values * u.km / u.s,
    W=df.vz.values * u.km / u.s,
    representation_type="cartesian",
    # Velocity representation
    differential_type="cartesian",
)
# transform to ICRS
icrs_coords = gal_coords.transform_to(ICRS())
icrs_coords.representation_type = "spherical"

ra = icrs_coords.ra.value
dec = icrs_coords.dec.value
dist = icrs_coords.distance.value
pmra = icrs_coords.pm_ra.value * np.cos(np.deg2rad(dec))
pmdec = icrs_coords.pm_dec.value
rv_calc = icrs_coords.radial_velocity.value

df['ra'] = ra
df['dec'] = dec
df['dist'] = dist
df['pmra'] = pmra
df['pmdec'] = pmdec
df['radial_velocity'] = rv_calc

df['vt_ra'] = df.pmra * df.dist * 4.74
df['vt_dec'] = df.pmdec * df.dist * 4.74

idx_cluster_1 = np.where(labels==1)[0]
idx_cluster_2 = np.where(labels==2)[0]
idx_bg = np.where(labels==0)[0]

In [ ]:
est = VrOpt(ra, dec, pmra, pmdec, dist)

 # Lower triangular matrix indices
a, b = np.tril_indices(len(X), -1)
_, _, delta_v = est.vr_opt(a, b)

# Create pairwise distance matrix with delta_v entries
dist_v_opt = np.zeros((len(X), len(X)))
dist_v_opt[a, b] = delta_v
dist_v_opt[b, a] = delta_v

# Compute semi-cohesion metric G
G = np.copy(dist_v_opt)
G = np.full_like(G, np.sum(G, axis=0)/len(G)) + np.full_like(G, np.sum(G, axis=0)/len(G)).T - np.sum(G)/len(G)**2 - G

## Functions for visualization

In [ ]:
def visualise_dataset_scatterplot_matrix(df, labels, colors, alphas, zorders):

    # I assume that the data for the background is given last in vectors: colors, alphas, zorders
    # Background settings
    bg_color = colors[-1]
    bg_alpha = alphas[-1]
    bg_zorder = zorders[-1]

    # Cluster settings
    cluster_colors = colors[:-1]
    cluster_alphas = alphas[:-1]
    cluster_zorders = zorders[:-1]


    palette = {'Background': bg_color}
    num_clusters = len(np.unique(labels)) - 1  # Exclude background
    cluster_labels = range(1, num_clusters + 1)
    for i, cluster_label in enumerate(cluster_labels):
        palette[f'Cluster {cluster_label}'] = cluster_colors[i]


    label_replacements = {
        'x': 'Position x',
        'y': 'Position y',
        'z': 'Position z',
        'vx': 'Velocity x',
        'vy': 'Velocity y',
        'vz': 'Velocity z',
        'vt_ra': 'Tangential Velocity Ra',
        'vt_dec': 'Tangential Velocity Dec'
    }

    features_set_1 = ['x', 'y', 'z', 'vx', 'vy', 'vz']
    features_set_2 = ['x', 'y', 'z', 'vt_ra', 'vt_dec']

    def plot_matrix(features, title):
      
        df_plot = df[features].copy()
        df_plot['label'] = 'Background' 

        # Assign cluster labels to the data points
        for cluster_label in cluster_labels:
            cluster_indices = np.where(labels == cluster_label)[0]
            df_plot.loc[cluster_indices, 'label'] = f'Cluster {cluster_label}'

        g = sns.pairplot(df_plot, vars=features, hue='label', palette=palette, corner=True, diag_kind="kde")

        # Use the given parameter for: alpha, zorder
        for ax_row in g.axes:
            for ax in ax_row:
                if ax:
                    xlabel = ax.get_xlabel()
                    ylabel = ax.get_ylabel()

                    if xlabel not in df_plot.columns or ylabel not in df_plot.columns:
                        continue

                    for label, color, alpha, zorder in zip(
                        ['Background'] + [f'Cluster {cl}' for cl in cluster_labels],
                        [bg_color] + cluster_colors,
                        [bg_alpha] + cluster_alphas,
                        [bg_zorder] + cluster_zorders):

                        subset = df_plot[df_plot['label'] == label]

                        # Add border to points
                        ax.scatter(
                            subset[xlabel],
                            subset[ylabel],
                            color=color,
                            alpha=alpha,
                            zorder=zorder,
                            edgecolor='white',  
                            linewidth=0.5,
                            label=label,
                        )

        # Customize axis labels
        for ax in g.axes.flat:
            if ax:
                xlabel = ax.get_xlabel()
                ylabel = ax.get_ylabel()
                if xlabel in label_replacements:
                    ax.set_xlabel(label_replacements[xlabel], fontsize=14)
                else:
                    ax.set_xlabel(xlabel, fontsize=14)
                if ylabel in label_replacements:
                    ax.set_ylabel(label_replacements[ylabel], fontsize=14)
                else:
                    ax.set_ylabel(ylabel, fontsize=14)

        sns.move_legend(g, "upper right", bbox_to_anchor=(0.9, 0.9), title="Membership:")
        legend = g._legend
        legend.get_title().set_fontsize(20)
        for text in legend.texts:
            text.set_fontsize(18)

        g.figure.suptitle(title, fontsize=25, y = 1.02)
        plt.show()

    # Plot two features sets
    plot_matrix(features_set_1, "Relationships between Position and Velocity in Spherical Coordinate")
    plot_matrix(features_set_2, "Relationships between Position and Tangential Velocity")


In [ ]:
#visualise_dataset_scatterplot_matrix(df, labels, colors, alphas, zorders)

In [ ]:
def visualise_dataset_2d(df, x_col, y_col, colors, alphas, zorders, labels):

    fig = plt.figure(figsize=(10, 10))

    # I assume that the data for the background is given last in vectors: colors, alphas, zorders
    # Background settings
    background_color = colors[-1]
    background_alpha = alphas[-1]
    background_zorder = zorders[-1]

    # Plot background
    idx_bg = np.where(labels == 0)[0]
    plt.scatter(df.loc[idx_bg, x_col], df.loc[idx_bg, y_col],
                s=3, c=background_color, alpha=background_alpha, 
                zorder=background_zorder, label="Background")

    # Plot clusters, exclude background parameters
    num_clusters = int(labels.max())
    cluster_colors = colors[:-1] 
    cluster_alphas = alphas[:-1] 
    cluster_zorders = zorders[:-1]

    for i in range(1, num_clusters + 1):
        cluster_indices = np.where(labels == i)[0]
        plt.scatter(df.loc[cluster_indices, x_col], df.loc[cluster_indices, y_col],
                    s=3, c=cluster_colors[i - 1], alpha=cluster_alphas[i - 1], 
                    zorder=cluster_zorders[i - 1], label=f"Cluster {i}")

    plt.xlabel(x_col, fontsize=13)
    plt.ylabel(y_col, fontsize=13)
    plt.title(f"Dataset in {x_col}, {y_col}", fontsize=20)
    plt.legend(title="membership ", title_fontsize=15, fontsize=13, markerscale=4)
    plt.show()

In [ ]:
#visualise_dataset_2d(
  #  df=df,
 #   x_col='x', 
 #   y_col='y',
 #   colors=colors,  
 #   alphas=alphas,  
 #   zorders=zorders,
  #  labels = labels
  #  )

In [ ]:
def visualise_dataset_3d(df, true_labels, colors, alphas):

    # I assume that the data for the background is given last in vectors: colors, alphas, zorders
    background_color = mcolors.to_hex(colors[-1])
    background_alpha = alphas[-1]

    # Convert cluster colors to HEX format
    cluster_colors = [mcolors.to_hex(c) for c in colors[:-1]]
    cluster_alphas = alphas[:-1]

    data = []

    # Add background points
    idx_background = true_labels == 0
    data.append(
        go.Scatter3d(
            x=df.loc[idx_background, 'x'],
            y=df.loc[idx_background, 'y'],
            z=df.loc[idx_background, 'z'],
            mode='markers',
            marker=dict(size=2, color=background_color, opacity=background_alpha),
            name='Background'
        )
    )

    # Add clusters based on true labels
    for cluster_id in np.unique(true_labels):
        if cluster_id != 0:  # Skip background points
            idx_cluster = true_labels == cluster_id
            data.append(
                go.Scatter3d(
                    x=df.loc[idx_cluster, 'x'],
                    y=df.loc[idx_cluster, 'y'],
                    z=df.loc[idx_cluster, 'z'],
                    mode='markers',
                    #marker_line = dict(width=0.1, color='white'),
                    marker=dict(
                        size=3, 
                        color=cluster_colors[int(cluster_id) - 1], 
                        opacity=cluster_alphas[int(cluster_id) - 1]
                        
                    ),
                    name=f'Cluster {int(cluster_id)}'
                )
            )

    fig = go.Figure(
        data=data,
        layout=go.Layout(
            margin=dict(l=0, r=0, b=0, t=0),
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z'
            ),
            legend=dict(itemsizing='constant', font=dict(size=16))
        )
    )

    fig.show()


In [ ]:
#visualise_dataset_3d(df, labels, colors, alphas)

In [ ]:
def visualise_cluster_result(df, cluster_labels, true_labels, K, x_col, y_col, colors, alphas, zorders):

    # Subplots for each detected cluster
    fig, ax = plt.subplots(1, K, figsize=(5 * K, 5), sharex=True, sharey=True)

    if K == 1:
        ax = [ax]

    # Background settings
    background_color = colors[-1]
    background_alpha = alphas[-1]
    background_zorder = zorders[-1]

    # Cluster settings, exclude background parameters
    cluster_colors = colors[:-1]  
    cluster_alphas = alphas[:-1]  
    cluster_zorders = zorders[:-1]  

    for l_i, axis in enumerate(ax):
        
        idx_sel = cluster_labels == l_i  # Points assigned to current detected cluster

        # Plot true background points
        idx_background = (true_labels == 0) & idx_sel # Assuming 0 is background true label

        axis.scatter(
            df.loc[idx_background, x_col],
            df.loc[idx_background, y_col],
            s=3,
            alpha=background_alpha,
            zorder=background_zorder,
            c=background_color,
            label="Background"
        )

        # Plot points for cluster using true labels' colors
        for idx, true_cluster_id in enumerate(np.unique(true_labels)):
            if true_cluster_id != 0:  # Skip background
                idx_true_cluster = (true_labels == true_cluster_id) & idx_sel
                axis.scatter(
                    df.loc[idx_true_cluster, x_col],
                    df.loc[idx_true_cluster, y_col],
                    s=3,
                    alpha=cluster_alphas[idx - 1],
                    zorder=cluster_zorders[idx - 1],
                    c=cluster_colors[idx - 1],  
                    label=f"True Cluster {int(true_cluster_id)}"
                )

        axis.set_xlabel(x_col, fontsize=15)
        axis.set_ylabel(y_col, fontsize=15)
        axis.set_title(f"Detected Cluster {l_i + 1}", fontsize=20)
        axis.legend(fontsize=12, markerscale = 4)

    plt.tight_layout()
    plt.show()

___

## Test with softmax clustering algorithm

In [ ]:
def softmax_clustering_optimized_iphd(Gamma, K, theta=0.001, epsilon=0.0001, max_iter=250, initial_partition=None):

    start_time = time.time()

    # Set g[i,i] = 0 for all i
    np.fill_diagonal(Gamma, 0)
    n = Gamma.shape[0]

    np.random.seed(42)

    # Use the initial partition if provided, otherwise initialize randomly
    if initial_partition is not None:
        p_i = initial_partition
    else:
        p_i = np.random.dirichlet(np.ones(K), size=n)

    z = np.zeros((n, K))  # for covariance

    for iteration in range(max_iter):
        # Compute covaraince (z)for all points and clusters
        
        z = Gamma @ p_i 
        # Stabilize softmax computation
        z_max = np.max(theta * z, axis=1, keepdims=True) 
        p_t = np.exp(theta * z - z_max) * p_i  

        #p_t = np.exp(theta * z) * p_i  # Softmax function
        p_i = p_t / (np.sum(p_t, axis=1, keepdims=True) + 1e-10)  # Update probability uisng softmax function and normalize it (it must sum to 1)

        theta += epsilon

         # If pi changes insignificantly, stop
        if iteration > 0 and np.allclose(p_i, prev_pi, atol=1e-6):
            break

        prev_pi = p_i.copy()
        
    execution_time = time.time() - start_time
    print(f'Execution time (softmax): {int(execution_time // 60)} minutes, {int(execution_time % 60)} seconds and {int((execution_time % 1) * 1000)} ms')
    print(f'Number of iterations (softmax): {iteration + 1}')

    return p_i, z

In [ ]:
#K = 6
#p_i_opt, z_opt = softmax_clustering_optimized_iphd(G, K)

In [ ]:
# get clusters from probability
def get_clusters(p_i):

    cluster_labels = np.argmax(p_i, axis=1)
    _, relabeled_clusters = np.unique(cluster_labels, return_inverse=True)
    
    return relabeled_clusters


In [ ]:
# get cluster labels and number of clusters
#labels_softmax = get_clusters(p_i_opt) 
#num_clusters = len(np.unique(labels_softmax))
#num_clusters

___

In [ ]:
#visualise_cluster_result(
 #   df=df,
 #   cluster_labels=labels_softmax,
#    true_labels=labels,
#    K=num_clusters,  
 #   x_col='x',
 #   y_col='y',
#    colors=colors,  
 #   alphas=alphas,  
 #   zorders=zorders  
#)

In [ ]:
def visualise_kdtree_partition_3d(df, node_indices):

    # Extract points
    points = df[['x', 'y', 'z']].values

    # Generate distinct color palette for all leaf nodes
    num_leaf_nodes = len(node_indices)
    rng = random.Random(42)
    colors = distinctipy.get_colors(num_leaf_nodes, rng=rng)
    colors_rgb = [distinctipy.get_hex(color) for color in colors]

    scatter_data = [
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(size=2, color='grey', opacity=0.5),
            name='Points'
        )
    ]

    # Add partitions for each leaf node
    for i, indices in enumerate(node_indices):
        node_points = points[indices]
        scatter_data.append(
            go.Scatter3d(
                x=node_points[:, 0],
                y=node_points[:, 1],
                z=node_points[:, 2],
                mode='markers',
                marker=dict(size=4, color=colors_rgb[i], opacity=0.8),
                name=f'Leaf Node {i + 1}'
            )
        )

    layout = go.Layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        showlegend=True
    )

    fig = go.Figure(data=scatter_data, layout=layout)
    fig.show()


---

# For testing partitions visualisation for tree

In [ ]:
def extract_leaf_nodes(kd_node, point_indices, points, node_indices, point_in_node):
  
    if isinstance(kd_node, KDTree.leafnode):
        unique_indices = np.setdiff1d(point_indices, list(point_in_node.keys()), assume_unique=True)
        if len(unique_indices) > 0:
            node_indices.append(unique_indices)
            point_in_node.update({idx: len(node_indices) - 1 for idx in unique_indices})
    else:
        split_dim = kd_node.split_dim
        split_val = kd_node.split
        left_indices = point_indices[points[point_indices, split_dim] <= split_val]
        right_indices = point_indices[points[point_indices, split_dim] > split_val]

        extract_leaf_nodes(kd_node.less, left_indices, points, node_indices, point_in_node)
        extract_leaf_nodes(kd_node.greater, right_indices, points, node_indices, point_in_node)
        
def prepare_kdtree(df, leafsize=300):
    points = df[['x', 'y', 'z']].values
    kd_tree = KDTree(points, leafsize=leafsize)

    node_indices = []
    point_in_node = {}
    extract_leaf_nodes(kd_tree.tree, np.arange(len(points)), points, node_indices, point_in_node)
    return kd_tree, node_indices, point_in_node

---

In [ ]:
#kd_tree, node_indices, point_in_node = prepare_kdtree(df, leafsize=300)

#visualise_kdtree_partition_3d(df, node_indices)

In [ ]:
# add planes representing partitions for each node
def add_partition_planes(node, bounds, depth=0, planes=[]):

    if isinstance(node, KDTree.leafnode):
        return  

    # Determine splitting dimension and value
    split_dim = depth % 3  #  0, 1, 2 for x, y, z
    split_val = node.split

    # Create plane for current split
    if split_dim == 0:  # Split along x-axis
        x_plane = np.full((10, 10), split_val)
        y_plane, z_plane = np.meshgrid(
            np.linspace(bounds[1][0], bounds[1][1], 10),
            np.linspace(bounds[2][0], bounds[2][1], 10)
        )
        planes.append((x_plane, y_plane, z_plane))
    elif split_dim == 1:  # Split along y-axis
        y_plane = np.full((10, 10), split_val)
        x_plane, z_plane = np.meshgrid(
            np.linspace(bounds[0][0], bounds[0][1], 10),
            np.linspace(bounds[2][0], bounds[2][1], 10)
        )
        planes.append((x_plane, y_plane, z_plane))
    elif split_dim == 2:  # Split along z-axis
        z_plane = np.full((10, 10), split_val)
        x_plane, y_plane = np.meshgrid(
            np.linspace(bounds[0][0], bounds[0][1], 10),
            np.linspace(bounds[1][0], bounds[1][1], 10)
        )
        planes.append((x_plane, y_plane, z_plane))


    left_bounds = [bound.copy() for bound in bounds]
    right_bounds = [bound.copy() for bound in bounds]
    left_bounds[split_dim][1] = split_val
    right_bounds[split_dim][0] = split_val

    # Repeat for child nodes
    add_partition_planes(node.less, left_bounds, depth + 1, planes)
    add_partition_planes(node.greater, right_bounds, depth + 1, planes)


def visualise_kdtree_partition_with_planes_3d(df, node_indices, kd_tree):

    # Extract points
    points = df[['x', 'y', 'z']].values

    # Generate a distinct color palette for all leaf nodes
    num_leaf_nodes = len(node_indices)
    rng = random.Random(42)
    colors = distinctipy.get_colors(num_leaf_nodes, rng =rng)
    colors_rgb = [distinctipy.get_hex(color) for color in colors]

    scatter_data = [
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(size=2, color='grey', opacity=0.5),
            name='Points'
        )
    ]

    # Add partitions for each leaf node
    for i, indices in enumerate(node_indices):
        node_points = points[indices]
        scatter_data.append(
            go.Scatter3d(
                x=node_points[:, 0],
                y=node_points[:, 1],
                z=node_points[:, 2],
                mode='markers',
                marker=dict(size=4, color=colors_rgb[i], opacity=0.8),
                name=f'Leaf Node {i + 1}'
            )
        )

    # Add splitting planes
    bounds = [
        [points[:, 0].min(), points[:, 0].max()],
        [points[:, 1].min(), points[:, 1].max()],
        [points[:, 2].min(), points[:, 2].max()],
    ]
    planes = []
    add_partition_planes(kd_tree.tree, bounds, planes=planes)

    for x_plane, y_plane, z_plane in planes:
        scatter_data.append(
            go.Surface(
                x=x_plane,
                y=y_plane,
                z=z_plane,
                opacity=0.3,  # plane transparency
                showscale=False,
                colorscale=[[0, 'black'], [1, 'black']],
                name='Partition Plane'
            )
        )

    layout = go.Layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        showlegend=True
    )

    fig = go.Figure(data=scatter_data, layout=layout)
    fig.show()

In [ ]:
#kd_tree, node_indices, point_in_node = prepare_kdtree(df, leafsize=300)

#visualise_kdtree_partition_with_planes_3d(df, node_indices, kd_tree)

In [ ]:
# Add partition lines for each node in KDTree in 2d plots

def add_partition_lines_2d(node, bounds, depth=0, lines=[]):

    if isinstance(node, KDTree.leafnode):
        return  # Stop at leaf nodes

    # Determine splitting dimension and value
    split_dim = depth % 2  # 0 for x-axis, 1 for y-axis
    split_val = node.split

    if split_dim == 0:  # Split along x-axis
        lines.append(((split_val, bounds[1][0]), (split_val, bounds[1][1])))  # vertical line
    elif split_dim == 1:  # Split along y-axis
        lines.append(((bounds[0][0], split_val), (bounds[0][1], split_val)))  # horizontal line

    # Update bounds for left and right children
    left_bounds = [bound.copy() for bound in bounds]
    right_bounds = [bound.copy() for bound in bounds]
    left_bounds[split_dim][1] = split_val
    right_bounds[split_dim][0] = split_val

    # Repeat for child nodes
    add_partition_lines_2d(node.less, left_bounds, depth + 1, lines)
    add_partition_lines_2d(node.greater, right_bounds, depth + 1, lines)


# Visualize  KDTree partitioning and dataset in 2D, showing points colored by their true labels
def visualise_kdtree_partition_2d(df, kd_tree, true_labels, colors, alphas, zorders):
  

    points = df[['x', 'y']].values

    # Background settings
    background_color = colors[-1]
    background_alpha = alphas[-1]
    background_zorder = zorders[-1]

    # Cluster settings
    cluster_colors = colors[:-1]
    cluster_alphas = alphas[:-1]
    cluster_zorders = zorders[:-1]

    # Map each label to its respective color, alpha, and zorder
    unique_labels = np.unique(true_labels)
    label_to_color = {}
    label_to_alpha = {}
    label_to_zorder = {}

    for i, label in enumerate(unique_labels):
        if label == 0:  # Background
            label_to_color[label] = background_color
            label_to_alpha[label] = background_alpha
            label_to_zorder[label] = background_zorder
        else:  # Clusters
            cluster_idx = (i - 1) % len(cluster_colors)
            label_to_color[label] = cluster_colors[cluster_idx]
            label_to_alpha[label] = cluster_alphas[cluster_idx]
            label_to_zorder[label] = cluster_zorders[cluster_idx]

    fig, ax = plt.subplots(figsize=(10, 10))

    for label in unique_labels:
        idx = true_labels == label
        ax.scatter(
            points[idx, 0],
            points[idx, 1],
            s=10,
            color=label_to_color[label],
            alpha=label_to_alpha[label],
            zorder=label_to_zorder[label],
            label=f'Cluster {int(label)}' if label != 0 else 'Background'
        )

    # Add partition lines
    bounds = [
        [points[:, 0].min(), points[:, 0].max()],
        [points[:, 1].min(), points[:, 1].max()],
    ]
    lines = []
    add_partition_lines_2d(kd_tree.tree, bounds, lines=lines)

    for line in lines:
        (x_start, y_start), (x_end, y_end) = line
        ax.plot([x_start, x_end], [y_start, y_end], color='black', alpha=1, linestyle='--', linewidth=2, zorder=0)

    # Number of partitions = number of lines
    num_partitions = len(lines)

    ax.set_title(f'KDTree Partitioning in 2D with Real Labels\nNumber of Partitions: {num_partitions + 1}', fontsize=16)
    ax.set_xlabel('X', fontsize=14)
    ax.set_ylabel('Y', fontsize=14)
    ax.legend(fontsize=12)
    plt.show()

In [ ]:
#visualise_kdtree_partition_2d(df, kd_tree, labels, colors, alphas, zorders)

In [ ]:
def visualise_leaves_partition_with_true_labels(df, node_indices, labels, colors, alphas, zorders):
 
    # Extract unique labels and map them real label parameters like color
    unique_labels = np.unique(labels)
    background_color = colors[-1]
    background_alpha = alphas[-1]  
    background_alpha = 0.6 # Higher alpha for background for better visibility
    background_zorder = zorders[-1]

    # Map labels to colors, alphas, and zorders
    label_to_color = {}
    label_to_alpha = {}
    label_to_zorder = {}

    for label in unique_labels:
        if label == 0:  # Background
            label_to_color[label] = background_color
            label_to_alpha[label] = background_alpha
            label_to_zorder[label] = background_zorder
        else:  # Clusters
            cluster_idx = int(label - 1)
            label_to_color[label] = colors[cluster_idx]
            label_to_alpha[label] = alphas[cluster_idx]
            label_to_zorder[label] = zorders[cluster_idx]

    for i, indices in enumerate(node_indices):
        leaf_points = df.iloc[indices][['x', 'y', 'z']].values
        leaf_labels = labels[indices]  # true labels for points in leaf

        # Points outside the partition
        outside_indices = np.setdiff1d(np.arange(len(df)), indices)
        outside_points = df.iloc[outside_indices][['x', 'y', 'z']].values

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        def plot_partition(ax, x_col, y_col):
            # Plot outside points in light grey
            ax.scatter(
                outside_points[:, x_col],
                outside_points[:, y_col],
                s=10,
                c='lightgrey',
                marker='x',
                alpha=0.3,
                zorder=1,
                label='Outside Points'
            )

            # Plot leaf points with true labels
            for label in np.unique(leaf_labels):
                label_mask = leaf_labels == label
                ax.scatter(
                    leaf_points[label_mask, x_col],
                    leaf_points[label_mask, y_col],
                    s=15,
                    color=label_to_color[label],
                    alpha=label_to_alpha[label],
                    zorder=label_to_zorder[label],
                    label=f"Cluster {int(label)}" if label != 0 else "Background"
                )

            # Add boundary lines for the partition
            x_min, x_max = leaf_points[:, x_col].min(), leaf_points[:, x_col].max()
            y_min, y_max = leaf_points[:, y_col].min(), leaf_points[:, y_col].max()

            # Draw boundary lines
            ax.plot([x_min, x_min], [y_min, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Left vertical
            ax.plot([x_max, x_max], [y_min, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black') # Right vertical
            ax.plot([x_min, x_max], [y_min, y_min], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Bottom horizontal
            ax.plot([x_min, x_max], [y_max, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Top horizontal

            ax.set_xlabel(['x', 'y', 'z'][x_col], fontsize=12)
            ax.set_ylabel(['x', 'y', 'z'][y_col], fontsize=12)
            ax.legend(fontsize=10, markerscale=1.5)
            ax.grid(True)

        plot_partition(axes[0], 0, 1)  # x-y
        plot_partition(axes[1], 0, 2)  # x-z
        plot_partition(axes[2], 1, 2)  # y-z

        plt.suptitle(f'Leaf {i + 1}: Partition Boundary with True Labels', fontsize=16)
        plt.tight_layout()
        plt.show()


In [ ]:
#kd_tree, node_indices, point_in_node = prepare_kdtree(df, leafsize=300)

#visualise_leaves_partition_with_true_labels(df, node_indices, labels, colors, alphas, zorders)

In [ ]:
def visualise_leaves_partition_with_cluster_labels(df, node_indices, similarity_matrix, colors, K_max=3, theta=0.001, epsilon=0.0001, max_iter=250):

    points = df[['x', 'y', 'z']].values

    for leaf_idx, indices in enumerate(node_indices):
        leaf_points = points[indices]  # Points within the current leaf
        leaf_similarity = similarity_matrix[np.ix_(indices, indices)]  # Subset similarity matrix for leaf

        # Outside points
        outside_indices = np.setdiff1d(np.arange(len(df)), indices)
        outside_points = points[outside_indices]

        # Softmax clustering for current leaf
        K = min(K_max, len(indices)) 
        softmax_probs, _ = softmax_clustering_optimized_iphd(leaf_similarity, K, theta, epsilon, max_iter)

        # Assign each point in leaf to cluster with the highest probability
        cluster_assignments = np.argmax(softmax_probs, axis=1)

        # Convert cluster ids to 1, 2, ... for current leaf
        unique_clusters = np.unique(cluster_assignments)
        cluster_id_map = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_clusters)}
        remapped_clusters = np.vectorize(cluster_id_map.get)(cluster_assignments)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        def plot_partition(ax, x_col, y_col):
            used_labels = [] 

            # Plot outside points in light grey 
            ax.scatter(
                outside_points[:, x_col],
                outside_points[:, y_col],
                s=10,
                c='lightgrey',
                marker='x',
                label='Outside Points',
                alpha=0.6,
            )

            # Plot each cluster's points
            for remapped_cluster_id in np.unique(remapped_clusters):
                cluster_mask = remapped_clusters == remapped_cluster_id
                if np.any(cluster_mask):  
                    color_index = remapped_cluster_id - 1  # Map cluster id to color
                    ax.scatter(
                        leaf_points[cluster_mask, x_col],
                        leaf_points[cluster_mask, y_col],
                        s=15,
                        color=colors[color_index],
                        alpha=0.9,
                        label=f"Cluster {remapped_cluster_id}" if remapped_cluster_id not in used_labels else None,
                    )
                    used_labels.append(remapped_cluster_id)


            # Add boundary lines for the partition
            x_min, x_max = leaf_points[:, x_col].min(), leaf_points[:, x_col].max()
            y_min, y_max = leaf_points[:, y_col].min(), leaf_points[:, y_col].max()

            # Draw boundary lines
            ax.plot([x_min, x_min], [y_min, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Left vertical
            ax.plot([x_max, x_max], [y_min, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black') # Right vertical
            ax.plot([x_min, x_max], [y_min, y_min], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Bottom horizontal
            ax.plot([x_min, x_max], [y_max, y_max], linestyle='--', alpha=0.8, zorder=0, color = 'black')  # Top horizontal


            ax.set_xlabel(['x', 'y', 'z'][x_col])
            ax.set_ylabel(['x', 'y', 'z'][y_col])
            ax.legend(fontsize=10, markerscale=2)
            ax.grid(True)

        plot_partition(axes[0], 0, 1)  # x-y
        plot_partition(axes[1], 0, 2)  # x-z
        plot_partition(axes[2], 1, 2)  # y-z

        plt.suptitle(f'Leaf {leaf_idx + 1}: Partition Boundary with Cluster Labels', fontsize=16)
        plt.tight_layout()
        plt.show()


In [ ]:
#cluster_colors = ['tab:pink', 'tab:brown', 'tab:cyan', 'tab:orange', 'tab:purple']  
#kd_tree, node_indices, point_in_node = prepare_kdtree(df, leafsize=300)

#visualise_leaves_partition_with_cluster_labels(df, node_indices, G, cluster_colors, K_max=6)